# 40 · LoRA / QLoRA 跑通第一次微调

> **学习目标**：理解 LoRA 核心参数 + 手撸最小 SFT 脚本（数据 / tokenizer / SFTTrainer / save），对比 r=4/8/16/32 四组 eval_loss 曲线。

> **预备**：39 号跑过（了解 HF 三件套 / chat template）。

> **为什么重要**：LoRA 是 PEFT（参数高效微调）的主流方法。**不查文档 30 分钟内写完最小 SFT 训练脚本**，是微调工作的基本门槛。

**注意**：本 notebook 所有训练代码用 OFFLINE 模拟。真实训练需要切换到 `ft` env 并安装 transformers / datasets / accelerate / peft / trl / bitsandbytes。

In [ ]:
MODE = 'OFFLINE'  # 'OFFLINE'（离线）或 'ONLINE'（在线，需要 ft env）

import json, hashlib, time
print(f'MODE = {MODE}')

## 1. PEFT 全景 —— Full FT / LoRA / QLoRA / Prefix Tuning / DoRA

**参数高效微调（PEFT）**的目标：**用最小参数增量，达到接近全量微调的质量**。

**全量微调 vs PEFT**：
- Full FT：微调所有参数 → 质量最好，显存 8-20GB（7B），训练慢
- LoRA：在原模型旁附加低秩矩阵 → 质量略低于 Full（<5%），显存 4-8GB
- QLoRA：4-bit 量化 + LoRA → 显存 6-12GB（7B），训练极慢
- Prefix Tuning：在 prompt 前加可训练向量 → 质量略好于 LoRA，显存 2-4GB

**LoRA vs QLoRA 的关系**：LoRA 可以和 QLoRA 一起用（QLoRA 的量化发生在 LoRA 之前）。

In [ ]:
# OFFLINE 模式：模拟不同 PEFT 方法的参数占用

def simulate_peft_params(base_params, method, rank=16, alpha=32, quant_bits=None):
    """模拟不同 PEFT 方法的参数增量（仅供参考）"""
    # fp16: 2 bytes/param
    fp16_params = base_params * 2  # 比如一个 7B 模型 = 14GB
    
    if method == 'full':
        # 全量微调：所有参数都可训练
        trainable = fp16_params
        trainable_gb = trainable / 2  # 假设输入已经是 fp16
    elif method == 'lora':
        # LoRA：只在指定 modules 添加低秩矩阵
        # 假设 trainables 是总参数的 10%
        trainable = fp16_params * 0.1
        trainable_gb = trainable / 2
    elif method == 'qlora':
        # QLoRA：4-bit 量化 + LoRA
        # 量化部分：显存 1/4（假设 chunk 量化）
        # LoRA 部分和普通 LoRA 一样
        trainable = fp16_params * 0.1
        trainable_gb = (fp16_params * 0.25) / 2 + (trainable / 2)  # 混合显存
    elif method == 'prefix':
        # Prefix：只可训练 prefix 向量（参数量极小）
        trainable = fp16_params * 0.001  # 0.1%
        trainable_gb = trainable / 2
    else:
        raise ValueError
    
    return {'base': fp16_params, 'trainable': trainable, 'trainable_gb': trainable_gb}

# 对比 7B 模型
base_7b = simulate_peft_params(7, 'full')
lora_7b = simulate_peft_params(7, 'lora', rank=16, alpha=32)
qlora_7b = simulate_peft_params(7, 'qlora', rank=16, alpha=32)
prefix_7b = simulate_peft_params(7, 'prefix')

print('=== 7B 模型不同 PEFT 方法参数对比 ===')
methods = {
    'Full FT': base_7b,
    'LoRA (r=16)': lora_7b,
    'QLoRA (r=16)': qlora_7b,
    'Prefix Tuning': prefix_7b,
}

for name, stats in methods.items():
    print(f'  {name:15s}: 训练 {stats["trainable_gb"]:.2f} GB (增量 {stats["trainable_gb"]/stats["base"]/2*100:.1f}% of base)')

## 2. LoRA 核心参数详解

**LoRA 的数学直觉**：
- 原始模型：`W = W0 + ΔW`，训练 ΔW（全量）
- LoRA：`W = W0 + B·A`，只训练 B 和 A（秩 r 的矩阵乘法）
- r 小 → 参数增量小；r 大 → 质量更接近全量

**关键超参**：
- `r`（rank）：秩，控制 LoRA 增量的大小（常用 8/16/32）
- `alpha`：缩放因子，通常 `alpha = 2 * r`（让 LoRA 增量不被过于放大）
- `target_modules`：哪些层应用 LoRA（常用 `q_proj` / `v_proj`）
- `dropout`：防止过拟合

**经验法则**：
- `r=8`：快速验证流程
- `r=16`：平衡（常用）
- `r=32`：接近全量，质量更好但显存更多

In [ ]:
# OFFLINE 模式：模拟不同 r 值对 eval_loss 的影响

def simulate_lora_train(r, alpha, base_loss=2.5):
    """模拟不同 r 的训练效果（仅作启发式）"""
    # 经验：r 越大，loss 越接近全量微调（损失越小）
    # 简单线性插值（仅为演示）
    improvement = (r / 32) * 0.15  # 最多 0.15 loss 改进
    new_loss = base_loss - improvement
    return new_loss

print('=== LoRA 不同 r 值对 eval_loss 的影响（7B，base_loss=2.5）===')
for r in [4, 8, 16, 32]:
    alpha = 2 * r  # 典型设置
    loss = simulate_lora_train(r, alpha)
    print(f'  r={r:2d}, alpha={alpha:3d} -> eval_loss = {loss:.3f} (改进 {2.5-loss:.3f})')

## 3. 最小 SFT 脚本（OFFLINE 模拟）

**SFT（Supervised Fine-Tuning）**：用标注好的 `(prompt, completion)` 数据训练模型。

**训练脚本骨架**：
1. 准备数据（JSONL / JSON）
2. 加载模型 + tokenizer
3. 用 `apply_chat_template` 转换对话格式
4. 用 `SFTTrainer` 训练
5. 保存 LoRA 权重

In [ ]:
# OFFLINE 模式：模拟最小 SFT 脚本骨架

# 1. 准备数据
SFT_DATA = [
    {
        'messages': [
            {'role': 'user', 'content': '解释什么是 LoRA。'},
            {'role': 'assistant', 'content': 'LoRA 是一种参数高效的微调方法，通过在原模型旁附加低秩矩阵来训练，只需更新少量参数。'}
        ]
    },
    {
        'messages': [
            {'role': 'user', 'content': 'Transformer 怎么工作？'},
            {'role': 'assistant', 'content': 'Transformer 基于注意力机制，将输入序列映射到隐藏状态，通过自注意力捕捉长距离依赖。'}
        ]
    },
    {
        'messages': [
            {'role': 'user', 'content': 'RAG 的优缺点？'},
            {'role': 'assistant', 'content': '优点：缓解幻觉，实时更新知识；缺点：检索误差传播，延迟稍高。'}
        ]
    }
]

print(f'共 {len(SFT_DATA)} 条 SFT 数据（OFFLINE 模拟）')
for i, d in enumerate(SFT_DATA[:2]):
    print(f'  [{i}] {len(d["messages"])} 轮对话')

In [ ]:
# 2. 加载模型 + tokenizer（OFFLINE stub）

class FakeModel:
    def __init__(self):
        self.params = '7B'
    def train(self):
        return 'model in training mode'

model = FakeModel()
print(f'模型: {model.params}')

In [ ]:
# 3. 用 apply_chat_template 转换对话格式（OFFLINE）

tokenizer = FakeTokenizer()

train_prompts = []
for item in SFT_DATA:
    prompt = tokenizer.apply_chat_template(
        item['messages'],
        tokenize=False,
        add_generation_prompt=False  # SFT 数据通常是「助手回答」，不用加 generation prompt
    )
    train_prompts.append(prompt)

print('=== SFT prompt（转换后）===')
for i, p in enumerate(train_prompts):
    print(f'\n[{i}]')
    print(p[:150] + '...' if len(p) > 150 else p)

In [ ]:
# 4. 模拟 SFTTrainer 训练（OFFLINE）

def simulate_sft_train(prompts, r, alpha, epochs=2, steps=10):
    """模拟 SFT 训练过程（仅作演示）"""
    print(f'开始 SFT 训练: r={r}, alpha={alpha}, epochs={epochs}, steps={steps}')
    print(f'训练数据: {len(prompts)} 条 prompt')
    
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0
        for step in range(steps):
            # 模拟 loss 下降
            loss = max(0.5, 2.5 * (1 - (epoch * steps + step) / (epochs * steps)))
            losses.append(loss)
            epoch_loss += loss
        print(f'  Epoch {epoch+1}: avg_loss = {epoch_loss/steps:.3f}')
    
    return losses

losses = simulate_sft_train(train_prompts, r=16, alpha=32, epochs=2, steps=10)

print(f'\n最终 eval_loss: {losses[-1]:.3f}')

In [ ]:
# 5. 对比 r=4/8/16/32 四组 loss 曲线（OFFLINE 模拟）

configs = [(4, 8), (8, 16), (16, 32), (32, 64)]

print('=== LoRA 不同 r 值的 eval_loss 曲线对比（OFFLINE 模拟）===')

all_losses = {}
for r, alpha in configs:
    losses = simulate_sft_train(train_prompts, r=r, alpha=alpha, epochs=2, steps=10)
    all_losses[r] = losses
    print(f'\nr={r}:')
    for epoch in range(2):
        start = epoch * 10
        end = (epoch + 1) * 10
        print(f'  Epoch {epoch+1}: {sum(all_losses[r][start:end])/10:.3f}')

## 4. LoRA merge + 量化导出步骤（文字版）

训练完成后，需要把 LoRA 权重合并到基座模型，并导出为完整模型或量化版本。

**步骤**：
1. **保存 LoRA 权重**：`trainer.save_model('./lora_output')`
2. **加载基座 + LoRA**：
   ```python
   from peft import PeftModel
   base_model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B')
   model = PeftModel.from_pretrained(base_model, './lora_output')
   ```
3. **合并 LoRA 权重**：
   ```python
   model = model.merge_and_unload()
   ```
4. **保存完整模型**：
   ```python
   model.save_pretrained('./merged_model')
   ```
5. **导出 GGUF**：
   ```bash
   python -m llama.cpp.convert-hf-to-gguf ./merged_model --outfile ./model.gguf --outtype q4_k_m
   ```

## 深入思考

1. **为什么 `alpha = 2 * r` 是常见设置？**
   - alpha 控制 LoRA 增量的缩放。太大 → 模型不稳定；太小 → LoRA 作用被削弱。2*r 是经验平衡。
2. **target_modules 选哪些层？**
   - 常用 `['q_proj', 'k_proj', 'v_proj', 'o_proj']`（attention）或 `['gate_proj', 'up_proj', 'down_proj']`（mlp）。最少 q_proj+v_proj 足够。
3. **r 越大越好吗？**
   - r=32 质量最好，但显存和训练时间都增加。实践中 r=16 是 sweet spot。
4. **LoRA merge 后还能再 LoRA 吗？**
   - 可以。多个 LoRA 可以依次 merge 到同一个 base，或者用 `mergekit` 一次性合并多个 LoRA。
5. **为什么 QLoRA 用 4-bit？**
   - 4-bit 可以大幅减少显存占用，让 7B/14B 模型能在消费级显卡上训练。精度损失相对可控。

**改一改**：
 - 在 REAL env 里，`SFTTrainer` 会自动处理 tokenization + padding + loss 计算
 - 真实训练需要 `dataset = Dataset.from_list(SFT_DATA)` + `tokenizer(dataset['messages'], ...)`
 - 加载真实模型：`model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B', device_map='auto')`

## 自检 ✅

- [ ] 默写 LoRA 核心参数（r / alpha / target_modules / dropout）
- [ ] 解释 LoRA 的数学直觉（W = W0 + BA）
- [ ] 解释「为什么 r 越大 eval_loss 越低」
- [ ] 列出 LoRA merge + GGUF 导出的 5 个关键步骤
- [ ] 解释「为什么 QLoRA 需要量化」

## 下一步

→ [`41_dpo_preference_optimization.ipynb`](41_dpo_preference_optimization.ipynb)